# 🚀 Darpan Labs - Train Digital Twin Adapters on Google Colab

This notebook trains all 18 persona adapters using **Mistral-7B-Instruct-v0.2**.

**⚡ NEW: Automatic DATA Generation & Drive Persistence**

**Why Mistral instead of Llama-2?**
- ✅ **No gated access** - Works immediately without approval
- ✅ **Better performance** - Often outperforms Llama-2-7b
- ✅ **Same size** - 7B parameters
- ✅ **Apache 2.0 license** - Fully open

**Requirements:**
- Google Colab Pro (for better GPU)
- HuggingFace token (free, no approval needed)
- GitHub repo: https://github.com/aniketm-dl/mvp_v1.0

**Runtime Settings:**
- Runtime > Change runtime type > GPU (T4, A100, or V100)
- High-RAM if available

## Step 1: Check GPU Availability

In [ ]:
!nvidia-smi

## Step 2: Mount Google Drive (Required for DATA persistence)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("\n✅ Google Drive mounted!")
print("   DATA will be saved to: /content/drive/MyDrive/darpan_mvp_data/")

## Step 3: Clone Repository from GitHub

In [ ]:
import os

# Clone from GitHub (no DATA folder in git)
!git clone https://github.com/aniketm-dl/mvp_v1.0.git

# Change to project directory
os.chdir('/content/mvp_v1.0')
!pwd

print("\n✅ Repository cloned!")
print("   Note: DATA folder not in git (too large)")
print("   We'll generate or sync it in the next step")

## Step 4: Install Dependencies

In [ ]:
!pip install -q torch transformers>=4.42.0 peft>=0.10.0 accelerate>=0.30.0 datasets>=2.20.0 bitsandbytes sentencepiece protobuf

print("\n✅ Dependencies installed!")

## Step 5: Authenticate with HuggingFace

Get your token from: https://huggingface.co/settings/tokens

**Note:** Mistral doesn't require special approval - any HuggingFace account works!

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Step 6: Verify Mistral-7B Access

In [ ]:
from transformers import AutoTokenizer

try:
    tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2")
    print("✅ Mistral-7B access verified!")
    print(f"✅ Tokenizer vocabulary size: {len(tokenizer)}")
    print("\n🎉 No approval needed - ready to train!")
except Exception as e:
    print(f"❌ Error: {e}")
    print("\nMake sure you used a valid HuggingFace token")

## Step 7: Setup Training Data (NEW!)

**⚡ This step is NEW and handles DATA automatically!**

The DATA folder (487MB) is not in the GitHub repo. This cell will:
1. Check if DATA exists in Google Drive
2. If yes: Copy from Drive (instant!)
3. If no: Generate all training data (~5-10 min) and save to Drive

**What gets generated:**
- `personas.json` (18 persona definitions)
- `twin_bank.json` (twin embeddings)
- `DATA/sft/` (2,700 training examples for 18 personas)
- `DATA/opera/` (optional ~476MB dataset from HuggingFace)

**This only runs once!** Future sessions will use the Drive copy.

In [ ]:
from pathlib import Path
import os
import time

print("=" * 70)
print("📁 CHECKING DATA FOLDER")
print("=" * 70)

# Define Drive DATA location
drive_data_path = Path("/content/drive/MyDrive/darpan_mvp_data")
local_data_path = Path("DATA")

# Check if DATA exists in Drive
if drive_data_path.exists() and (drive_data_path / "personas.json").exists():
    print("\n✅ Found existing DATA in Google Drive!")
    print(f"   Location: {drive_data_path}")
    print("\n📋 Syncing from Drive to Colab workspace...")
    
    # Copy from Drive to local
    !cp -r /content/drive/MyDrive/darpan_mvp_data DATA/
    
    # Verify
    if local_data_path.exists() and (local_data_path / "personas.json").exists():
        print("✅ DATA synced successfully!")
        
        # Show summary
        sft_files = list((local_data_path / "sft").glob("*.jsonl")) if (local_data_path / "sft").exists() else []
        print(f"\n📊 DATA Summary:")
        print(f"   • Personas: {len(sft_files)} files")
        print(f"   • Ready for training!")
    else:
        print("❌ Sync failed - will regenerate")
        !rm -rf DATA
        drive_data_exists = False
    
    drive_data_exists = True
else:
    print("\n📝 DATA not found in Drive - will generate fresh data")
    drive_data_exists = False

# If DATA doesn't exist in Drive, generate it
if not drive_data_exists:
    print("\n" + "=" * 70)
    print("🔧 GENERATING TRAINING DATA")
    print("=" * 70)
    print("\nThis will take ~5-10 minutes and includes:")
    print("   1. Download OPeRA dataset from HuggingFace (~476MB)")
    print("   2. Generate 18 persona definitions")
    print("   3. Generate 2,700 training examples (150 per persona)")
    print("\n⏱️  Starting in 3 seconds...")
    time.sleep(3)
    
    # Run the setup script
    !python scripts/colab/setup_training_data.py
    
    # Save to Drive for next time
    if local_data_path.exists():
        print("\n" + "=" * 70)
        print("💾 SAVING DATA TO GOOGLE DRIVE")
        print("=" * 70)
        print(f"\nSaving to: {drive_data_path}")
        
        # Create Drive directory and copy
        !mkdir -p /content/drive/MyDrive/darpan_mvp_data
        !cp -r DATA/* /content/drive/MyDrive/darpan_mvp_data/
        
        print("\n✅ DATA saved to Google Drive!")
        print("   Future sessions will load instantly from Drive")
    else:
        print("\n❌ DATA generation failed - see errors above")
        raise Exception("DATA generation failed")

print("\n" + "=" * 70)
print("✨ DATA SETUP COMPLETE!")
print("=" * 70)
print("\n📁 DATA folder ready:")
!du -sh DATA/
print("\n⏭️  Ready for training!")

## Step 8: Verify Training Data

In [ ]:
import os
import json
from pathlib import Path

# List training data files
sft_dir = Path("DATA/sft")
if sft_dir.exists():
    files = list(sft_dir.glob("*.jsonl"))
    print(f"✅ Found {len(files)} training data files:")
    
    total_examples = 0
    for f in sorted(files):
        lines = len(f.read_text().strip().split('\n'))
        total_examples += lines
        size_kb = f.stat().st_size / 1024
        print(f"   • {f.name:<30} {lines:>3} examples ({size_kb:>6.1f}KB)")
    
    print(f"\n📊 Total: {total_examples:,} training examples ready!")
else:
    print("❌ DATA/sft directory not found!")
    print("   Something went wrong in Step 7 - please re-run that cell")

## Step 9: Train a Single Twin (Test Run)

Test with one persona first to ensure everything works.

In [ ]:
# Test with bargain_hunter first
print("🧪 Test Training: The Bargain Hunter")
print("   Model: Mistral-7B-Instruct-v0.2")
print("   Expected time: 5-15 minutes\n")

!python scripts/train_llm_persona_sft.py \
  --twin_id bargain_hunter \
  --base_model mistralai/Mistral-7B-Instruct-v0.2 \
  --epochs 1 \
  --max_length 512 \
  --lr 2e-4

## Step 10: Verify Test Training

In [ ]:
adapter_file = Path("artifacts/llm_adapters/bargain_hunter/adapter_model.safetensors")
if adapter_file.exists():
    size_mb = adapter_file.stat().st_size / (1024 * 1024)
    print(f"✅ Test training successful! Adapter: {size_mb:.1f}MB")
    print(f"\n🎉 Mistral training works! Ready for full training.")
else:
    print("❌ Test training failed - adapter not created")
    print("   Check errors above before proceeding")

## Step 11: Train ALL 18 Personas

⚠️ **This will take 2-5 hours depending on GPU**

Colab Pro GPU estimates:
- T4: ~10-15 min per twin = 3-4 hours total
- V100/A100: ~5-8 min per twin = 1.5-2.5 hours total

**Mistral typically trains slightly faster than Llama-2!**

In [ ]:
import time
start_time = time.time()

print("🚀 Starting full training for all 18 personas...\n")
!python scripts/train_all_adapters.py

elapsed = (time.time() - start_time) / 60
print(f"\n⏱️ Total training time: {elapsed:.1f} minutes ({elapsed/60:.1f} hours)")

## Step 12: Verify All Adapters

In [ ]:
import json
from pathlib import Path

# Load personas
personas = json.loads(Path("DATA/personas.json").read_text())["personas"]

print("📊 Adapter Training Summary:\n")
print(f"{'Twin ID':<25} {'Status':<10} {'Size (MB)':<12}")
print("=" * 50)

total_size = 0
success_count = 0

for persona in personas:
    twin_id = persona["id"]
    adapter_file = Path(f"artifacts/llm_adapters/{twin_id}/adapter_model.safetensors")
    
    if adapter_file.exists():
        size_mb = adapter_file.stat().st_size / (1024 * 1024)
        total_size += size_mb
        success_count += 1
        print(f"{twin_id:<25} {'✅ Success':<10} {size_mb:>10.1f}")
    else:
        print(f"{twin_id:<25} {'❌ Missing':<10} {'-':>10}")

print("=" * 50)
print(f"\n✅ Successfully trained: {success_count}/{len(personas)}")
print(f"💾 Total adapter size: {total_size:.1f}MB")

if success_count == len(personas):
    print(f"\n🎉 All {len(personas)} Mistral-powered twins ready!")

## Step 13: Package Adapters for Download

In [ ]:
# Create a zip file of all trained adapters
print("📦 Creating adapter package...\n")
!cd artifacts && zip -rq llm_adapters_mistral.zip llm_adapters/

# Check size
import os
zip_size = os.path.getsize("artifacts/llm_adapters_mistral.zip") / (1024 * 1024)
print(f"\n✅ Package created: llm_adapters_mistral.zip ({zip_size:.1f}MB)")

## Step 14: Save to Google Drive

In [ ]:
# Copy to Google Drive for easy download
!cp artifacts/llm_adapters_mistral.zip /content/drive/MyDrive/

print("✅ Adapters saved to Google Drive: MyDrive/llm_adapters_mistral.zip")
print("\nYou can now:")
print("1. Download from Google Drive to your local machine")
print("2. Extract to: artifacts/llm_adapters/")
print("   Command: python scripts/colab/download_adapters.py --zip ~/Downloads/llm_adapters_mistral.zip --backup")
print("3. Test locally: python interact_cli.py")
print("\n🎉 Mistral-powered digital twins ready to use!")

## Step 15: Test an Adapter (Optional)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

print("🧪 Testing The Bargain Hunter adapter...\n")

# Load base model
base_model = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(base_model)
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Load adapter
twin_id = "bargain_hunter"
model = PeftModel.from_pretrained(model, f"artifacts/llm_adapters/{twin_id}")

# Test prompt
prompt = "[INST] You are The Bargain Hunter. Be concise. 1-2 sentences. Should I buy this $50 product or wait for a sale? [/INST]"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    temperature=0.3,
    top_p=0.9,
    do_sample=True
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n" + "="*60)
print("BARGAIN HUNTER (Mistral-powered):")
print(response.split("[/INST]")[-1].strip())
print("="*60)

## 🎉 Training Complete!

### What you have now:
- ✅ 18 Mistral-7B fine-tuned adapters
- ✅ DATA folder persisted in Google Drive
- ✅ Better performance than Llama-2
- ✅ No licensing restrictions
- ✅ Ready for deployment

### Next steps:
1. Download `llm_adapters_mistral.zip` from Google Drive
2. Extract locally:
   ```bash
   python scripts/colab/download_adapters.py \
     --zip ~/Downloads/llm_adapters_mistral.zip \
     --backup
   ```
3. Test: `python interact_cli.py`
4. Deploy: `uvicorn src.api.service:app --reload`

### ⚡ NEW: DATA Persistence
- Your DATA folder (~487MB) is saved in Google Drive
- Future training sessions will load instantly (no regeneration!)
- Location: `/content/drive/MyDrive/darpan_mvp_data/`

### Advantages of Mistral over Llama-2:
- 🚀 **Faster training** - Slightly more efficient
- 💬 **Better instruction following** - More natural responses
- 🔓 **No gates** - Instant access, no approval wait
- 📜 **Apache 2.0** - Fully permissive license